In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [10]:
df = pd.read_csv(
    Path.cwd().parent / 'data' /
    'UK_energy_build__2025_2030__with_coordinates.csv'
)
df.head()

,Year,Department,Asset_Name,Asset_Type,Capacity_MW,Latitude,Longitude,Start_Latitude,Start_Longitude,End_Latitude,End_Longitude,Location_Type,Notes,Sources
0,2025,Offshore wind,Moray West,Offshore wind farm,882,58.094000,-2.988000,NaN,NaN,NaN,NaN,point,Coordinates from Wikipedia (Moray West Wind Fa...,https://en.wikipedia.org/wiki/Moray_West_Wind_...
1,2025,Offshore wind,Neart na Gaoithe (NnG),Offshore wind farm,448,56.270000,-2.250000,NaN,NaN,NaN,NaN,point,Coordinates from Global Energy Monitor (GEM).,https://www.gem.wiki/Neart_Na_Gaoithe_wind_farm
2,2025,Offshore wind,Dogger Bank B (Creyke Beck B),Offshore wind farm,1200,54.979972,1.679972,NaN,NaN,NaN,NaN,point,Coordinates from TheWindPower.net page for Dog...,https://www.thewindpower.net/windfarm_en_16773...
3,2026,Offshore wind,Dogger Bank C (Teesside A),Offshore wind farm,1200,55.039972,2.819972,NaN,NaN,NaN,NaN,point,Coordinates from TheWindPower.net page for Dog...,https://www.thewindpower.net/windfarm_en_16689...
4,2026,Offshore wind,Sofia (Dogger Bank Teesside B),Offshore wind farm,1400,55.210000,2.330000,NaN,NaN,NaN,NaN,point,Coordinates from Global Energy Monitor (Sofia ...,https://www.gem.wiki/Sofia_wind_farm


In [19]:
fleet = pd.read_csv(
    Path.cwd().parent / 'data' /
    'gb_2024_capacities.csv', index_col=0
)
fleet.head()

,Installed_Capacity_GW,Notes,Source_URL
Technology,,,
Onshore wind,15.3,GB onshore wind capacity end-2024 (Electric In...,https://reports.electricinsights.co.uk/wp-cont...
Offshore wind,14.8,GB offshore wind capacity end-2024 (Electric I...,https://reports.electricinsights.co.uk/wp-cont...
Solar PV,17.2,GB solar PV capacity end-2024 (Electric Insights),https://reports.electricinsights.co.uk/wp-cont...
Battery storage (BESS),4.7,Total battery capacity in GB (end 2024),https://modoenergy.com/research/gb-battery-ene...
Pumped-storage hydro (PSH),2.8,Operational pumped hydro capacity in GB (4 sch...,https://www.drax.com/wp-content/uploads/2024/0...


In [ ]:
import pypsa
import geopandas as gpd
import warnings
# warnings.filterwarning('ignore')

future = '2028'

n = pypsa.Network(
    Path.cwd().parent / 'results' / '2024-03-22' / 'network_flex_s_nodal.nc'
)

onshore = gpd.read_file(Path.cwd().parent / 'data' / 'regions_onshore_s.geojson')
offshore = gpd.read_file(Path.cwd().parent / 'data' / 'regions_offshore_s.geojson')



c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatibl

In [52]:
n.buses.country

Bus
8838                    GB
8013                    GB
8649                    GB
4950                    GB
4951                    GB
                  ...     
Denmark            Denmark
France              France
Netherlands    Netherlands
Belgium            Belgium
Norway              Norway
Name: country, Length: 292, dtype: object

In [67]:
from shapely.geometry import Point


def adjust_to_future_year(
    n,
    year,
    current_fleet,
    future_additions,
    regions_onshore,
    regions_offshore,
    ):

    n = n.copy()

    if year == 'off':
        return
    
    year = int(year)
    
    assert n.snapshots[0].year == 2024, 'model year has to be 2024 for future simulation'

    print('Warning! Check for changes in fossil, nuclear and biomass fleet')

    # Assets are treated differently; iterating over one-by-one assets first
    
    ##### Offshore Wind #####

    # taking fleet strength not from model but dataset because the model only knows
    # weather-dependent actual generation capacity during that day
    current_capacity = fleet.loc['Offshore wind', 'Installed_Capacity_GW']

    new_assets = future_additions.loc[
        (future_additions.Year <= year) &
        (future_additions.Department == 'Offshore wind')
    ]
    pypsa_carrier = 'offwind'
    network_capacity = pd.concat((
        n.generators.loc[gen, 'p_nom'] * (
            n.generators_t.p_max_pu[gen] 
            if gen in n.generators_t.p_max_pu.columns 
            else pd.Series(1, index=n.generators_t.p_max_pu.index)
        )
        for gen in n.generators.index[n.generators.carrier == pypsa_carrier]
    ), axis=1).sum(axis=1)

    print(network_capacity)

    buslocs = n.buses.loc[n.buses.country == 'GB', ['x', 'y']]

    for name, row in new_assets.iterrows():

        pt = row[['Longitude', 'Latitude']].values

        # Calculate distances from point to all bus locations
        distances = np.sqrt(
            (buslocs['x'] - pt[0])**2 + 
            (buslocs['y'] - pt[1])**2
        )
        
        # Get index of minimum distance
        closest_bus = distances.idxmin()
        print(f"Closest bus is {closest_bus}")

        print(row)

        new_capacity_share = row['Capacity_MW'] / 1000 / current_capacity
        new_capacity = network_capacity * new_capacity_share

        print('nameplate', row['Capacity_MW'] / 1000)
        print('current', current_capacity)
        print('new_capacity_share', new_capacity_share)
        print('new_capacity', new_capacity)

        p_nom = new_capacity.max()
        p_max_pu = new_capacity / p_nom

        print(p_nom)
        print(p_max_pu)


        # implement a simple check which bus is closest, these are voronoi cells and this will result in the same outcome
        print('=====================')
        break




In [69]:
n.generators.carrier.unique()

array(['onwind', 'offwind', 'solar', 'nuclear', 'fossil', 'biomass',
       'local_market'], dtype=object)

In [70]:
adjust_to_future_year(
    n,
    future,
    fleet,
    df,
    onshore,
    offshore
)



c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  attrs.loc[bool_b, "default"] = attrs.loc[bool_b].isin({True, "True"})
c:\Users\lukas\miniforge3\envs\pypsa-earth\lib\site-packages\pypsa\components.py:318: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatibl

Warning! Check for changes in fossil, nuclear and biomass fleet
snapshot
2024-03-22 00:00:00    10675.583333
2024-03-22 00:30:00    10672.566667
2024-03-22 01:00:00    10737.666667
2024-03-22 01:30:00    10766.116667
2024-03-22 02:00:00    10753.600000
2024-03-22 02:30:00    10777.966667
2024-03-22 03:00:00    10932.850000
2024-03-22 03:30:00    10900.516667
2024-03-22 04:00:00    10743.300000
2024-03-22 04:30:00    10624.833333
2024-03-22 05:00:00    10511.783333
2024-03-22 05:30:00    10341.800000
2024-03-22 06:00:00    10381.850000
2024-03-22 06:30:00    10201.450000
2024-03-22 07:00:00    10094.150000
2024-03-22 07:30:00    10053.700000
2024-03-22 08:00:00    10095.750000
2024-03-22 08:30:00    10001.566667
2024-03-22 09:00:00     9891.466667
2024-03-22 09:30:00     9587.983333
2024-03-22 10:00:00     9585.050000
2024-03-22 10:30:00     9365.666667
2024-03-22 11:00:00     9109.216667
2024-03-22 11:30:00     8871.750000
2024-03-22 12:00:00     8523.333333
2024-03-22 12:30:00     813

In [64]:
n.generators.loc['BEATO-1']

bus                           6443
p_nom                        117.0
carrier                    offwind
marginal_cost                  0.0
control                         PQ
type                              
p_nom_extendable             False
p_nom_min                      0.0
p_nom_max                      inf
p_min_pu                       0.0
p_max_pu                       1.0
p_set                          0.0
q_set                          0.0
sign                           1.0
marginal_cost_quadratic        0.0
build_year                       0
lifetime                       inf
capital_cost                   0.0
efficiency                     1.0
committable                  False
start_up_cost                  0.0
shut_down_cost                 0.0
min_up_time                      0
min_down_time                    0
up_time_before                   1
down_time_before                 0
ramp_limit_up                  NaN
ramp_limit_down                NaN
ramp_limit_start_up 

In [72]:
fleet

,Installed_Capacity_GW,Notes,Source_URL
Technology,,,
Onshore wind,15.3,GB onshore wind capacity end-2024 (Electric In...,https://reports.electricinsights.co.uk/wp-cont...
Offshore wind,14.8,GB offshore wind capacity end-2024 (Electric I...,https://reports.electricinsights.co.uk/wp-cont...
Solar PV,17.2,GB solar PV capacity end-2024 (Electric Insights),https://reports.electricinsights.co.uk/wp-cont...
Battery storage (BESS),4.7,Total battery capacity in GB (end 2024),https://modoenergy.com/research/gb-battery-ene...
Pumped-storage hydro (PSH),2.8,Operational pumped hydro capacity in GB (4 sch...,https://www.drax.com/wp-content/uploads/2024/0...


In [28]:
df.loc[(df.Year <= 2026) & (df.Department == 'Offshore wind')]

,Year,Department,Asset_Name,Asset_Type,Capacity_MW,Latitude,Longitude,Start_Latitude,Start_Longitude,End_Latitude,End_Longitude,Location_Type,Notes,Sources
0,2025,Offshore wind,Moray West,Offshore wind farm,882,58.094000,-2.988000,NaN,NaN,NaN,NaN,point,Coordinates from Wikipedia (Moray West Wind Fa...,https://en.wikipedia.org/wiki/Moray_West_Wind_...
1,2025,Offshore wind,Neart na Gaoithe (NnG),Offshore wind farm,448,56.270000,-2.250000,NaN,NaN,NaN,NaN,point,Coordinates from Global Energy Monitor (GEM).,https://www.gem.wiki/Neart_Na_Gaoithe_wind_farm
2,2025,Offshore wind,Dogger Bank B (Creyke Beck B),Offshore wind farm,1200,54.979972,1.679972,NaN,NaN,NaN,NaN,point,Coordinates from TheWindPower.net page for Dog...,https://www.thewindpower.net/windfarm_en_16773...
3,2026,Offshore wind,Dogger Bank C (Teesside A),Offshore wind farm,1200,55.039972,2.819972,NaN,NaN,NaN,NaN,point,Coordinates from TheWindPower.net page for Dog...,https://www.thewindpower.net/windfarm_en_16689...
4,2026,Offshore wind,Sofia (Dogger Bank Teesside B),Offshore wind farm,1400,55.210000,2.330000,NaN,NaN,NaN,NaN,point,Coordinates from Global Energy Monitor (Sofia ...,https://www.gem.wiki/Sofia_wind_farm


In [18]:
fleet

,Technology,Installed_Capacity_GW,Notes,Source_URL
0,Onshore wind,15.3,GB onshore wind capacity end-2024 (Electric In...,https://reports.electricinsights.co.uk/wp-cont...
1,Offshore wind,14.8,GB offshore wind capacity end-2024 (Electric I...,https://reports.electricinsights.co.uk/wp-cont...
2,Solar PV,17.2,GB solar PV capacity end-2024 (Electric Insights),https://reports.electricinsights.co.uk/wp-cont...
3,Battery storage (BESS),4.7,Total battery capacity in GB (end 2024),https://modoenergy.com/research/gb-battery-ene...
4,Pumped-storage hydro (PSH),2.8,Operational pumped hydro capacity in GB (4 sch...,https://www.drax.com/wp-content/uploads/2024/0...
